# 01b — Generate AI Reviews with GPT-5 mini (OpenAI API)

**Run once.** Reads the human review pool from Drive, samples 2,500 unique products, and prompts `gpt-5-mini` to write a customer-style review for each. Output is appended to a shared `ai_reviews.csv` on Drive (the `generator` column distinguishes models). Resume-safe: already-generated products are skipped.

- Input: `MyDrive/ECS111FinalProject/data/raw/human_reviews.csv`
- Output: `MyDrive/ECS111FinalProject/data/generated/ai_reviews.csv`
- Cost: roughly \$1–3 of OpenAI API usage. No GPU needed.

**API notes (GPT-5 series is a reasoning model):**
- Use `max_completion_tokens`, not `max_tokens`.
- `temperature` is **not supported** — omit it.
- `reasoning_effort` accepts `minimal` / `low` / `medium` / `high`. We use `minimal`: review writing is a generation task, not a reasoning task, and minimal effort keeps latency/cost down and avoids reasoning tokens eating the completion budget.
- `verbosity` (`low`/`medium`/`high`) controls output length; we leave it default and steer length via the prompt.

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ECS111FinalProject'
HUMAN_PATH = os.path.join(PROJECT_DIR, 'data', 'raw', 'human_reviews.csv')
OUT_PATH = os.path.join(PROJECT_DIR, 'data', 'generated', 'ai_reviews.csv')
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
assert os.path.exists(HUMAN_PATH), f'missing {HUMAN_PATH} — run 01_collect_human_reviews first'
print('Input :', HUMAN_PATH)
print('Output:', OUT_PATH)

In [ ]:
!pip install -q openai pandas tqdm

In [ ]:
from getpass import getpass
os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key: ')

## Config

In [ ]:
MODEL = 'gpt-5-mini'
GENERATOR_LABEL = 'gpt5mini'
N = 2500
SEED = 42
WORKERS = 10
REASONING_EFFORT = 'minimal'

COLS = ['review_id', 'category', 'parent_asin', 'rating', 'title',
        'text', 'timestamp', 'label', 'generator']

## Prompt construction

Target word count is sampled from the human review length distribution so AI reviews can't be told apart by length alone.

In [ ]:
import random

PROMPT_TEMPLATE = (
    'Write a {rating}-star Amazon product review for: {product_title}\n'
    'Category: {category}\n'
    'Length: about {target_word_count} words.\n'
    'Voice: a real customer who bought this product. No emojis, no hashtags.\n'
    'Output only the review body — no title, no preamble.'
)


def sample_word_count(rng, human_lengths):
    return rng.choice(human_lengths)


def build_prompt(product_title, category, rating, target_word_count):
    return PROMPT_TEMPLATE.format(
        product_title=product_title or 'this product',
        category=str(category).replace('_', ' '),
        rating=int(round(rating)) if rating else 5,
        target_word_count=target_word_count,
    )

## Generation — parallel API calls, resume-safe, incremental writes

In [ ]:
import csv
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
rng = random.Random(SEED)

human_df = pd.read_csv(HUMAN_PATH)
human_lengths = human_df['text'].str.split().str.len().tolist()
seeds_df = (
    human_df.drop_duplicates(subset='parent_asin')
            .sample(n=N, random_state=SEED)
            .reset_index(drop=True)
)

# Resume: skip products already generated for this generator
done = set()
if Path(OUT_PATH).exists():
    prev = pd.read_csv(OUT_PATH)
    done = set(prev.loc[prev['generator'] == GENERATOR_LABEL, 'parent_asin'].astype(str))
    if done:
        print(f'resume: {len(done)} already done for {GENERATOR_LABEL}, skipping those')
seeds_df = seeds_df[~seeds_df['parent_asin'].astype(str).isin(done)].reset_index(drop=True)
print(f'to generate: {len(seeds_df)}')


def generate_one(prod, target_wc):
    prompt = build_prompt(prod['title'], prod['category'], prod['rating'], target_wc)
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            # budget covers both reasoning + visible tokens; minimal effort keeps reasoning small
            max_completion_tokens=int(target_wc * 4) + 256,
            reasoning_effort=REASONING_EFFORT,
        )
        text = (resp.choices[0].message.content or '').strip()
        if not text:
            return {'_error': 'empty completion', 'parent_asin': prod['parent_asin']}
    except Exception as e:
        return {'_error': str(e), 'parent_asin': prod['parent_asin']}

    return {
        'review_id': f"{GENERATOR_LABEL}_{prod['parent_asin']}_{SEED}",
        'category': prod['category'],
        'parent_asin': prod['parent_asin'],
        'rating': prod['rating'],
        'title': prod['title'],
        'text': text,
        'timestamp': '',
        'label': 1,
        'generator': GENERATOR_LABEL,
    }

In [ ]:
if seeds_df.empty:
    print('nothing to do — all products already generated.')
else:
    file_exists = Path(OUT_PATH).exists()
    f = open(OUT_PATH, 'a', newline='', encoding='utf-8')
    writer = csv.DictWriter(f, fieldnames=COLS, quoting=csv.QUOTE_MINIMAL)
    if not file_exists:
        writer.writeheader()
        f.flush()
    write_lock = threading.Lock()

    n_done, n_err = 0, 0
    pbar = tqdm(total=len(seeds_df), desc=f'{GENERATOR_LABEL} (workers={WORKERS})')

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures = [
            ex.submit(generate_one, prod.to_dict(), sample_word_count(rng, human_lengths))
            for _, prod in seeds_df.iterrows()
        ]
        for fut in as_completed(futures):
            row = fut.result()
            pbar.update(1)
            if row is None:
                continue
            if '_error' in row:
                n_err += 1
                pbar.set_postfix(errors=n_err)
                continue
            with write_lock:
                writer.writerow(row)
                f.flush()
            n_done += 1

    pbar.close()
    f.close()
    print(f'\ndone: wrote {n_done} new reviews to {OUT_PATH} (errors: {n_err})')

## Quick sanity check

In [ ]:
df = pd.read_csv(OUT_PATH)
sub = df[df['generator'] == GENERATOR_LABEL]
print(f'{GENERATOR_LABEL}: {len(sub)} reviews')
wc = sub['text'].str.split().str.len()
print(f'word count: min={wc.min()}, max={wc.max()}, mean={wc.mean():.1f}')
print('\nsample:\n')
print(sub['text'].iloc[0])